# Realtime Prediction V2 — Agentic

Modular Colab entry point for the realtime application.

Agents:
- Market Data Agent
- Sentiment Agent
- Data Quality Agent
- Forecast Agent
- OpenAI LLM Orchestrator

The trained ML model produces the numerical probability; the LLM only orchestrates and explains.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install -r '/content/drive/MyDrive/BTC reproducible multimodal pipeline/realtime_agentic_app/requirements.txt'

In [ ]:
from google.colab import userdata
import os, sys
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/BTC reproducible multimodal pipeline')
APP = PROJECT / 'realtime_agentic_app'
SENTIMENT_SCRIPT = Path('/content/drive/MyDrive/btc sentiment year data/btc_sentiment_agents.py')

if str(APP) not in sys.path:
    sys.path.insert(0, str(APP))

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = None

try:
    ALPHA_VANTAGE_API_KEY = userdata.get('ALPHA_VANTAGE_API_KEY')
except Exception:
    ALPHA_VANTAGE_API_KEY = None

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if ALPHA_VANTAGE_API_KEY:
    os.environ['ALPHA_VANTAGE_API_KEY'] = ALPHA_VANTAGE_API_KEY

print('OpenAI configured:', bool(OPENAI_API_KEY))
print('Alpha Vantage configured:', bool(ALPHA_VANTAGE_API_KEY))

In [ ]:
from realtime_pipeline import RealtimeCryptoPipeline

pipeline = RealtimeCryptoPipeline(
    project_root=PROJECT,
    sentiment_script=SENTIMENT_SCRIPT,
    alpha_vantage_key=ALPHA_VANTAGE_API_KEY,
)
live = pipeline.refresh()
live

In [ ]:
import pandas as pd
pd.DataFrame([
    pipeline.get_forecast(1),
    pipeline.get_forecast(6),
    pipeline.get_forecast(24),
])[['horizon_hours','direction','probability_up','model','feature_set','current_price']]

In [ ]:
pipeline.get_system_health()

In [ ]:
pipeline.get_latest_sentiment()

In [ ]:
from agents.llm_orchestrator import CryptoLLMOrchestrator

if OPENAI_API_KEY:
    agent = CryptoLLMOrchestrator(pipeline.tool_map())
    print(agent.ask(
        'Using the current data, what is the 24-hour BTC outlook? '
        'Explain the model signal and historical reliability.'
    ))
else:
    print('Add OPENAI_API_KEY in Colab Secrets first.')

## Optional Gradio UI

In [ ]:
# %cd "/content/drive/MyDrive/BTC reproducible multimodal pipeline/realtime_agentic_app"
# !python app_gradio.py